# 여긴어때 · 10반 3조
목적지 근처 서울시 공영주차장을 조건에 맞춰 최대 3곳 추천합니다.

이 노트북만 Jupyter 또는 Colab에 올리고 위에서부터 실행합니다. Python 3.11 이상을 사용합니다.
외부 Python 파일, CSV, `.env` 파일은 필요하지 않습니다. API 키는 아래에서 입력합니다.

`src/`의 구현을 우선하며, 설계서는 참고 자료로만 사용합니다.
P2~P6의 함수·클래스 본문은 현재 프로젝트에서 옮겼습니다. 모델은 조에서 정한 `gpt-5.6-luna`를 사용합니다(`src/format.py`에 남아 있는 구형 기본값도 노트북에서는 통일). 상대 import, 사용하지 않는 파일 경로와 P3의 즉시 실행 예제만 제외했습니다. 인자가 다른 부분은 P1 연결 셀에서 맞춥니다.
레퍼런스의 구조화 출력, `prompt | model | parser`, `RunnableLambda` 패턴을 사용합니다.


## 1. 설치와 설정


In [1]:
%pip install -q "langchain>=1,<2" "langchain-openai>=1,<2" requests


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from getpass import getpass

for key in ("OPENAI_API_KEY", "KAKAO_REST_API_KEY", "SEOUL_OPENAPI_KEY"):
    if not os.getenv(key):
        os.environ[key] = getpass(f"{key}: ")

# P2와 P6에서 같은 모델을 사용합니다.
os.environ["MODEL_NAME"] = "gpt-5.6-luna"
os.environ["OPENAI_MODEL"] = "gpt-5.6-luna"
os.environ["PARKING_AGENT_NO_LLM"] = "0"

# 동명 장소를 가까운 순으로 보여주기 위한 실습용 기준 위치입니다.
# 실제 현재 위치가 아니므로 필요한 좌표로 바꿉니다.
USER_LAT, USER_LNG = 37.4979, 127.0276

# 현재 P3에는 LANDMARKS가 없어 P2의 규칙 폴백용으로 빈 목록을 둡니다.
LANDMARKS = {}


## 2. 공통 자료형과 실행 컨텍스트 (P1)


In [3]:
"""파이프라인 전 구간의 데이터 계약입니다.

이 파일은 P1(계약·통합 담당)만 수정합니다.
다른 담당자는 읽기만 하며, 필드 추가·변경이 필요하면 조 채널에 먼저 제안합니다.
모든 모듈은 여기 정의된 타입만 주고받으며, dict를 그대로 넘기지 않습니다.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime
from typing import Literal

# --------------------------------------------------------------------------
# 공통 상수
# --------------------------------------------------------------------------

SortBy = Literal["distance", "price"]
DayType = Literal["weekday", "weekend", "holiday"]

#: 필수조건 탈락 사유입니다. 문자열을 임의로 만들지 않고 이 목록만 사용합니다.
RejectReason = Literal[
    "만차",
    "영업 종료",
    "영업시간 부족",
    "예산 초과",
    "거리 초과",
]


# --------------------------------------------------------------------------
# 1단계 · 사용자 입력에서 추출한 파라미터 (P2 산출)
# --------------------------------------------------------------------------


@dataclass
class RankingParams:
    """사용자 발화에서 추출한 검색 조건입니다.

    place만 필수이며 나머지는 미지정(None)일 수 있습니다.
    미지정 값을 임의의 기본값으로 채우지 않습니다. 기본값 적용은
    evaluate/rank 단계에서 수행하고, 적용 사실을 응답에 명시합니다.
    """

    place: str
    duration_minutes: int | None = None
    budget_won: int | None = None
    sort_by: SortBy = "distance"

    #: 직전 턴의 조건을 병합했는지 여부입니다. 리랭킹 응답 문구 분기에 사용합니다.
    merged_from_previous: bool = False


# --------------------------------------------------------------------------
# 2단계 · 앱이 주입하는 실행 컨텍스트 (P1 산출)
# --------------------------------------------------------------------------


@dataclass
class SearchPolicy:
    """검색 규칙입니다. LLM이 변경할 수 없는 코드 상수입니다."""

    adjacent_district_count: int = 5
    max_distance_m: int = 2000
    top_k: int = 3
    default_duration_minutes: int = 60


@dataclass
class RequestContext:
    """호출 1회 동안 고정되는 값입니다.

    대화 메시지에 넣지 않습니다. 위치정보가 대화 로그에 남지 않도록 하기 위함입니다.
    """

    user_id: str
    request_time: datetime  # KST
    day_type: DayType
    user_lat: float | None = None
    user_lng: float | None = None
    response_mode: Literal["driving", "normal"] = "normal"
    policy: SearchPolicy = field(default_factory=SearchPolicy)


# --------------------------------------------------------------------------
# 3단계 · 목적지 확정 (P3 산출)
# --------------------------------------------------------------------------


@dataclass
class Place:
    """지오코딩 결과 후보 1건입니다."""

    name: str
    address: str
    lat: float
    lng: float
    district: str  # 예: "강남구"
    #: 사용자 현재 위치로부터의 거리입니다. 위치 미제공 시 None입니다.
    distance_from_user_m: int | None = None


@dataclass
class GeocodeResult:
    """지오코딩 도구의 반환값입니다.

    예외를 던지지 않습니다. 실패도 정상 반환값으로 표현합니다.
    - candidates가 0건이면 message에 폴백 안내 문구가 담깁니다.
    - candidates가 2건 이상이면 호출자가 사용자에게 재확인합니다.
    """

    candidates: list[Place]
    message: str | None = None

    @property
    def is_confirmed(self) -> bool:
        return len(self.candidates) == 1


# --------------------------------------------------------------------------
# 4단계 · 주차장 후보 (P4 산출)
# --------------------------------------------------------------------------


@dataclass
class ParkingLot:
    """서울시 공영주차장 1건의 원본 정보입니다.

    확인할 수 없는 값은 추정하지 않고 None으로 둡니다.
    """

    code: str
    name: str
    address: str
    district: str
    lat: float | None = None
    lng: float | None = None

    total_slots: int | None = None
    current_cars: int | None = None

    #: "HHMM" 4자리 문자열입니다. 예: "0900". 24시간 운영은 "0000"/"2400"입니다.
    open_time: str | None = None
    close_time: str | None = None

    base_fee: int | None = None
    base_minutes: int | None = None
    extra_fee: int | None = None
    extra_minutes: int | None = None

    @property
    def available_slots(self) -> int | None:
        """잔여 주차면입니다. 원본 값이 없으면 None(확인 불가)입니다."""
        if self.total_slots is None or self.current_cars is None:
            return None
        return max(0, self.total_slots - self.current_cars)


@dataclass
class SearchResult:
    """주차장 검색 도구의 반환값입니다."""

    lots: list[ParkingLot]
    searched_districts: list[str]
    message: str | None = None


# --------------------------------------------------------------------------
# 5단계 · 판정과 계산 (P5 산출)
# --------------------------------------------------------------------------


@dataclass
class Evaluation:
    """주차장 1건에 대한 계산·판정 결과입니다.

    계산할 수 없는 항목은 None으로 두고 note에 사유를 적습니다.
    """

    lot: ParkingLot
    distance_m: int
    is_open: bool
    #: 마감까지 남은 분입니다. 24시간 운영이면 None입니다.
    minutes_until_close: int | None = None
    estimated_fee: int | None = None
    #: 요금 계산 근거입니다. 예: "기본 30분 1,000원 + 추가 90분 3,000원"
    fee_basis: str | None = None
    #: 할인 등 반영하지 못한 조건입니다.
    fee_note: str | None = None
    #: 기본값을 적용한 항목명입니다. 예: ["duration_minutes"]
    assumed_fields: list[str] = field(default_factory=list)


@dataclass
class Rejection:
    """필수조건을 만족하지 못해 제외된 후보입니다."""

    lot_name: str
    reasons: list[RejectReason]


@dataclass
class EvaluationResult:
    """판정 도구의 반환값입니다."""

    passed: list[Evaluation]
    rejected: list[Rejection]


# --------------------------------------------------------------------------
# 6단계 · 최종 추천 (P6 산출)
# --------------------------------------------------------------------------


@dataclass
class Recommendation:
    """응답에 노출되는 주차장 1건입니다.

    응답 문장의 모든 수치는 이 객체에서만 가져옵니다.
    출력 가드레일이 답변과 이 객체를 대조합니다.
    """

    rank: int
    name: str
    distance_m: int
    #: "3,200원" 또는 "계산 불가"
    fee_text: str
    #: "12면" 또는 "확인 불가"
    availability_text: str
    #: "24시간" 또는 "22:00 마감 (3시간 20분 남음)"
    hours_text: str
    reason: str = ""


@dataclass
class RankResult:
    """랭킹 도구의 반환값입니다. 0건이어도 예외를 던지지 않습니다."""

    recommendations: list[Recommendation]
    rejected: list[Rejection]
    sort_by: SortBy
    assumed_fields: list[str] = field(default_factory=list)

    @property
    def is_empty(self) -> bool:
        return len(self.recommendations) == 0


# --------------------------------------------------------------------------
# 파이프라인 최종 산출물
# --------------------------------------------------------------------------


@dataclass
class AgentResponse:
    """사용자에게 전달되는 최종 결과입니다."""

    answer: str
    params: RankingParams
    rank_result: RankResult | None = None
    #: 출력 가드레일 판정입니다. "SAFE" 또는 "UNSAFE"입니다.
    verdict: str = "SAFE"
    verdict_reason: str | None = None


In [4]:
"""실행 컨텍스트를 구성합니다. [담당: P1]

위치와 시각은 이곳에서만 읽습니다. 도구 내부에서 datetime.now()나
GPS를 직접 호출하지 않습니다.
"""

from __future__ import annotations

import os
from datetime import datetime, timedelta, timezone


KST = timezone(timedelta(hours=9))

#: 2026년 공휴일입니다. 필요한 범위만 등록합니다.
HOLIDAYS_2026 = {"2026-09-24", "2026-09-25", "2026-09-26", "2026-10-03", "2026-10-09"}


def resolve_day_type(dt: datetime) -> DayType:
    """기준 시각의 요일 구분을 반환합니다."""
    if dt.strftime("%Y-%m-%d") in HOLIDAYS_2026:
        return "holiday"
    return "weekend" if dt.weekday() >= 5 else "weekday"


def build_context(
    user_id: str = "demo-user",
    user_lat: float | None = None,
    user_lng: float | None = None,
    now: datetime | None = None,
) -> RequestContext:
    """호출 1회분의 컨텍스트를 만듭니다.

    now를 명시하면 테스트에서 시각을 고정할 수 있습니다.
    """
    request_time = now or datetime.now(KST)
    return RequestContext(
        user_id=user_id,
        request_time=request_time,
        day_type=resolve_day_type(request_time),
        user_lat=user_lat,
        user_lng=user_lng,
        response_mode=os.getenv("RESPONSE_MODE", "normal"),  # type: ignore[arg-type]
        policy=SearchPolicy(),
    )


def is_llm_disabled() -> bool:
    """규칙 기반 모드 여부입니다. 단위 테스트는 항상 이 모드로 실행합니다."""
    return os.getenv("PARKING_AGENT_NO_LLM") == "1"


## 3. 입력 이해와 입력 가드레일 (P2)
장소·시간·예산·정렬을 슬롯별로 추출합니다. 발화에 신호가 있는 슬롯만 LLM을 호출하며, 검증 실패 시 해당 슬롯을 재요청합니다. 호출 예외 시 규칙 추출로 폴백합니다.


In [5]:
"""사용자 발화에서 검색 조건을 추출합니다. [담당: P2]

발화의 슬롯(장소·시간·예산·정렬)마다 독립된 도구를 둡니다.
신호가 있는 슬롯만 LLM에 요청하고, 없으면 호출을 생략합니다.
LLM 슬롯 호출을 우선 사용하고, 예외 상황에서만 규칙 도구로 폴백합니다.
미지정 항목을 임의의 기본값으로 채우지 않습니다. None으로 두십시오.
"""

from __future__ import annotations

import os
import re

from pydantic import BaseModel, Field


try:
    from langchain_core.tools import tool as _lc_tool
except ImportError:  # pragma: no cover

    def _lc_tool(fn):
        """langchain 없이도 규칙 경로가 동작하도록 통과시킵니다."""
        return fn


PRICE_KEYWORDS = ("저렴", "싼", "싸게", "가격", "요금", "비싸")

MINUTES_PER_HOUR = 60
WON_PER_MANWON = 10_000
WON_PER_CHEONWON = 1_000

#: "한시간", "두시간" 같은 표현을 지원합니다.
KOREAN_HOURS = {
    "한": 1,
    "두": 2,
    "세": 3,
    "네": 4,
    "다섯": 5,
    "여섯": 6,
    "일곱": 7,
    "여덟": 8,
    "아홉": 9,
}

#: 장소 접미사입니다. P3의 LANDMARKS에 없는 이름도 geocode까지 전달되도록
#: "구"를 포함합니다. ("은평구" → geocode의 "지원하지 않는 장소" 안내)
PLACE_SUFFIXES = r"역|구청|구|동|로|길|점|몰|공원|타워|시장|백화점"

LLM_MODEL_DEFAULT = "gpt-5.6-luna"
LLM_TIMEOUT_SECONDS = 10

#: 슬롯 검증 실패 시 같은 슬롯을 재요청하는 횟수입니다. 규칙 폴백이 아닙니다.
LLM_MAX_ATTEMPTS = 2


# --------------------------------------------------------------------------
# 규칙 도구 4종: 결정론적 추출입니다. 예외 상황에서만 폴백으로 씁니다.
# --------------------------------------------------------------------------


def extract_place(utterance: str) -> str:
    """장소 이름을 추출합니다. 목적지 지명이 필요할 때 호출하십시오.

    발화에서 장소를 찾을 때만 호출하십시오. 시간·예산·정렬 판단에는
    호출하지 마십시오. 못 찾으면 빈 문자열을 둡니다.

    Args:
        utterance: 사용자 발화 원문입니다.
    """
    for name in sorted(LANDMARKS, key=len, reverse=True):
        if name in utterance:
            return name
    if m := re.search(r"([가-힣A-Za-z0-9]{2,10})\s*(?:근처|주변|인근|앞)", utterance):
        return m.group(1)
    return _suffix_place(utterance)


def _suffix_place(utterance: str) -> str:
    """접미사 토큰을 찾습니다. 조사 오탐("2시간으로"의 로 등)은 제외합니다."""
    for m in re.finditer(rf"([가-힣A-Za-z0-9]+(?:{PLACE_SUFFIXES}))", utterance):
        if _is_place_token(m.group(1)):
            return m.group(1)
    return ""


def _is_place_token(token: str) -> bool:
    """접미사 매칭이 조사 오탐이 아닌지 봅니다."""
    if token.endswith("으로"):
        return False
    for suffix in ("구", "동", "로", "길"):
        if token.endswith(suffix) and len(token) < len(suffix) + 2:
            return False
    return True


def extract_duration(utterance: str) -> int | None:
    """주차 시간을 분 단위로 추출합니다. 시간 표현 해석이 필요할 때 호출하십시오.

    시간 언급이 있는 발화에만 호출하십시오. 장소·예산·정렬 판단에는
    호출하지 마십시오. 없으면 None을 둡니다.

    Args:
        utterance: 사용자 발화 원문입니다.
    """
    if m := re.search(r"(\d+)\s*시간\s*반", utterance):
        return int(m.group(1)) * MINUTES_PER_HOUR + MINUTES_PER_HOUR // 2
    if m := re.search(r"(한|두|세|네|다섯|여섯|일곱|여덟|아홉)\s*시간\s*반", utterance):
        return KOREAN_HOURS[m.group(1)] * MINUTES_PER_HOUR + MINUTES_PER_HOUR // 2
    if m := re.search(r"(\d+)\s*시간", utterance):
        return int(m.group(1)) * MINUTES_PER_HOUR
    if m := re.search(r"(한|두|세|네|다섯|여섯|일곱|여덟|아홉)\s*시간", utterance):
        return KOREAN_HOURS[m.group(1)] * MINUTES_PER_HOUR
    if "반시간" in utterance:
        return MINUTES_PER_HOUR // 2
    if m := re.search(r"(\d+)\s*분", utterance):
        return int(m.group(1))
    return None


def extract_budget(utterance: str) -> int | None:
    """예산 상한을 원 단위로 추출합니다. 금액 표현 해석이 필요할 때 호출하십시오.

    금액 언급이 있는 발화에만 호출하십시오. 장소·시간·정렬 판단에는
    호출하지 마십시오. 없으면 None을 둡니다.

    Args:
        utterance: 사용자 발화 원문입니다.
    """
    if m := re.search(r"(\d+)\s*만\s*원", utterance):
        return int(m.group(1)) * WON_PER_MANWON
    if m := re.search(r"(\d+)\s*천\s*원", utterance):
        return int(m.group(1)) * WON_PER_CHEONWON
    if m := re.search(r"(\d[\d,]*)\s*원", utterance):
        return int(m.group(1).replace(",", ""))
    if "만원" in utterance:
        return WON_PER_MANWON
    if "천원" in utterance:
        return WON_PER_CHEONWON
    return None


def extract_sort(utterance: str) -> SortBy:
    """정렬 기준을 추출합니다. 가격 정렬 의도 확인이 필요할 때 호출하십시오.

    정렬 의도 판정에만 호출하십시오. 장소·시간·예산 판단에는
    호출하지 마십시오. 가격 언급이 있을 때만 price입니다.

    Args:
        utterance: 사용자 발화 원문입니다.
    """
    return "price" if any(k in utterance for k in PRICE_KEYWORDS) else "distance"


# --------------------------------------------------------------------------
# LLM 슬롯 4종: 슬롯별 집중 추출입니다. 우선 경로로 씁니다.
# --------------------------------------------------------------------------


class _PlaceSlot(BaseModel):
    """장소 슬롯의 구조화 출력입니다. reasoning을 먼저 채웁니다."""

    reasoning: str = Field(default="", description="추출 과정을 단계별로 적은 메모입니다.")
    place: str = Field(default="", description="핵심 지명만 둡니다.")


class _DurationSlot(BaseModel):
    """시간 슬롯의 구조화 출력입니다. reasoning을 먼저 채웁니다."""

    reasoning: str = Field(default="", description="추출 과정을 단계별로 적은 메모입니다.")
    duration_minutes: int | None = Field(default=None, description="주차 시간(분)입니다.")


class _BudgetSlot(BaseModel):
    """예산 슬롯의 구조화 출력입니다. reasoning을 먼저 채웁니다."""

    reasoning: str = Field(default="", description="추출 과정을 단계별로 적은 메모입니다.")
    budget_won: int | None = Field(default=None, description="예산 상한(원)입니다.")


class _SortSlot(BaseModel):
    """정렬 슬롯의 구조화 출력입니다. reasoning을 먼저 채웁니다."""

    reasoning: str = Field(default="", description="추출 과정을 단계별로 적은 메모입니다.")
    sort_by: str = Field(default="distance", description="정렬 의도가 있을 때만 price입니다.")


_PLACE_PROMPT = (
    "발화에서 목적지 지명을 찾습니다. "
    "근처·주변·주차장 같은 말은 떼고 핵심 지명만 둡니다. "
    "없으면 빈 문자열입니다. 먼저 reasoning에 과정을 적으십시오."
)
_DURATION_PROMPT = (
    "발화에서 주차 시간을 찾아 분으로 둡니다. 시간은 60을 곱합니다. "
    "언급이 없으면 None이며 값을 지어내지 않습니다. 먼저 reasoning에 과정을 적으십시오."
)
_BUDGET_PROMPT = (
    "발화에서 예산 상한을 찾아 원으로 둡니다. 만원은 10000을 곱합니다. "
    "언급이 없으면 None이며 값을 지어내지 않습니다. 먼저 reasoning에 과정을 적으십시오."
)
_SORT_PROMPT = (
    "싼 곳·저렴한 순 같은 정렬 의도가 있을 때만 price이고 아니면 distance입니다. "
    "가격 불만 표현(너무 비싸, 비싸다)도 정렬 의도로 봅니다. "
    "예산 언급만으로는 distance를 둡니다. 먼저 reasoning에 과정을 적으십시오."
)

_SLOT_SCHEMAS = {
    "place": (_PlaceSlot, _PLACE_PROMPT),
    "duration": (_DurationSlot, _DURATION_PROMPT),
    "budget": (_BudgetSlot, _BUDGET_PROMPT),
    "sort": (_SortSlot, _SORT_PROMPT),
}


def _call_slot_llm(slot: str, utterance: str, feedback: str | None = None) -> BaseModel | None:
    """슬롯 1개를 LLM에 1회 요청합니다.

    한 번에 한 슬롯만 다룹니다. 다른 슬롯이 필요하면 호출하지 마십시오.
    구조화 실패는 그대로 돌려주어 검증 단계가 다룹니다.
    예외 상황(미설치·키 없음·호출 실패)이면 None을 반환합니다.

    Args:
        slot: "place", "duration", "budget", "sort" 중 하나입니다.
        utterance: 사용자 발화 원문입니다.
        feedback: 검증 지적 사항입니다. 재요청 때만 씁니다.
    """
    try:
        from langchain_openai import ChatOpenAI

        schema, prompt = _SLOT_SCHEMAS[slot]
        model = ChatOpenAI(
            model=os.getenv("MODEL_NAME", LLM_MODEL_DEFAULT),
            temperature=0,
            timeout=LLM_TIMEOUT_SECONDS,
        )
        if feedback:
            prompt = f"{prompt}\n이전 추출 문제점: {feedback}\n위 문제를 고쳐 다시 추출하십시오."
        return model.with_structured_output(schema).invoke(f"{prompt}\n발화: {utterance}")
    except Exception:
        return None


def _validate_place(utterance: str, result: _PlaceSlot) -> list[str]:
    """장소 슬롯이 발화에 근거하는지 봅니다. 값을 고치지 않고 판정만 합니다."""
    place = _clean_place(result.place)
    if place and place.replace(" ", "") not in utterance.replace(" ", ""):
        return [f"장소 '{place}'가 발화에 없습니다"]
    return []


def _validate_duration(utterance: str, result: _DurationSlot) -> list[str]:
    """시간 슬롯이 발화에 근거하는지 봅니다. 값을 고치지 않고 판정만 합니다."""
    if result.duration_minutes is None:
        return []
    if result.duration_minutes <= 0:
        return ["주차 시간이 0 이하입니다"]
    if not _has_time_expression(utterance):
        return ["시간 언급이 없는데 주차 시간이 있습니다"]
    return []


def _validate_budget(utterance: str, result: _BudgetSlot) -> list[str]:
    """예산 슬롯이 발화에 근거하는지 봅니다. 값을 고치지 않고 판정만 합니다."""
    if result.budget_won is None:
        return []
    if result.budget_won < 0:
        return ["예산이 음수입니다"]
    if not _has_money_expression(utterance):
        return ["금액 언급이 없는데 예산이 있습니다"]
    return []


def _validate_sort(utterance: str, result: _SortSlot) -> list[str]:
    """정렬 슬롯이 발화에 근거하는지 봅니다. 값을 고치지 않고 판정만 합니다."""
    if result.sort_by not in ("price", "distance"):
        return ["정렬 값이 price/distance가 아닙니다"]
    if result.sort_by == "price" and not _has_sort_intent(utterance):
        return ["정렬 의도 언급이 없는데 price입니다"]
    return []


_SLOT_VALIDATORS = {
    "place": _validate_place,
    "duration": _validate_duration,
    "budget": _validate_budget,
    "sort": _validate_sort,
}


def _validate_slot(slot: str, utterance: str, result: BaseModel) -> list[str]:
    """슬롯 검증기로 판정만 합니다. 비어 있으면 정상입니다."""
    return _SLOT_VALIDATORS[slot](utterance, result)


def _has_time_expression(utterance: str) -> bool:
    """시간 언급이 있는지 봅니다."""
    if "반시간" in utterance:
        return True
    if re.search(r"\d+\s*시간", utterance):
        return True
    if re.search(r"(한|두|세|네|다섯|여섯|일곱|여덟|아홉)\s*시간", utterance):
        return True
    return re.search(r"\d+\s*분", utterance) is not None


def _has_money_expression(utterance: str) -> bool:
    """금액 언급이 있는지 봅니다."""
    return "원" in utterance or "예산" in utterance


def _has_sort_intent(utterance: str) -> bool:
    """정렬 의도 언급이 있는지 봅니다."""
    return any(k in utterance for k in PRICE_KEYWORDS)


def _has_place_signal(utterance: str) -> bool:
    """장소 언급이 있는지 봅니다. 없으면 place 슬롯을 생략합니다."""
    for name in LANDMARKS:
        if name in utterance:
            return True
    if re.search(r"[가-힣A-Za-z0-9]{2,10}\s*(?:근처|주변|인근|앞)", utterance):
        return True
    return _suffix_place(utterance) != ""


def _needed_slots(utterance: str) -> list[str]:
    """LLM 호출이 필요한 슬롯만 고릅니다.

    규칙 게이트이며 추출이 아닙니다. 신호가 없는 슬롯은 호출하지 않고
    생략값(place ""·duration None·budget None·sort "distance")으로 둡니다.
    생략값은 _merge에서 직전 조건 유지로 해석됩니다.
    """
    slots: list[str] = []
    if _has_place_signal(utterance):
        slots.append("place")
    if _has_time_expression(utterance):
        slots.append("duration")
    if _has_money_expression(utterance):
        slots.append("budget")
    if _has_sort_intent(utterance):
        slots.append("sort")
    return slots


#: 마지막 `_extract_by_llm` 호출에서 실제 요청한 슬롯 목록입니다.
#: 재시도도 1회로 셉니다. 턴당 호출 수 확인용 진단 값이며,
#: 단일 스레드 데모·테스트에서만 읽습니다.
LAST_SLOT_CALLS: list[str] = []


def _clean_place(place: str) -> str:
    """LLM이 붙인 잔여 접미사(근처·주차장 등)를 걷어냅니다."""
    cleaned = place.strip()
    cleaned = re.sub(r"\s*(?:근처|주변|인근|앞|주차장)+\s*$", "", cleaned)
    return cleaned.strip()


# --------------------------------------------------------------------------
# 조립: 계약 인터페이스입니다. 시그니처를 바꾸지 않습니다.
# --------------------------------------------------------------------------


def extract_params(
    utterance: str,
    prev: RankingParams | None = None,
) -> RankingParams:
    """발화를 RankingParams로 변환합니다.

    prev가 있으면 이번 발화에 명시된 필드만 덮어쓰고 나머지는 유지합니다.
    """
    if is_llm_disabled():
        params = _extract_by_rule(utterance)
    else:
        params = _extract_by_llm(utterance) or _extract_by_rule(utterance)

    if prev is not None:
        params = _merge(prev, params)
    return params


def _extract_by_llm(utterance: str) -> RankingParams | None:
    """LLM 슬롯 호출로 추출합니다. LLM 우선 경로입니다.

    신호가 있는 슬롯만 호출합니다. 생략된 슬롯은 생략값으로 두어
    _merge에서 직전 조건 유지로 해석됩니다.
    슬롯 1개라도 예외 상황이면 None을 반환해 턴 전체를 규칙으로 폴백합니다.
    슬롯별 검증 문제는 같은 슬롯을 재요청해 교정합니다.
    호출 내역은 LAST_SLOT_CALLS에 남깁니다. 예외를 던지지 않습니다.
    """
    global LAST_SLOT_CALLS
    called: list[str] = []
    slots: dict[str, BaseModel] = {}
    for slot in _needed_slots(utterance):
        result = _call_slot_llm(slot, utterance)
        called.append(slot)
        if result is None:
            LAST_SLOT_CALLS = called
            return None
        if issues := _validate_slot(slot, utterance, result):
            retry = _call_slot_llm(slot, utterance, feedback="; ".join(issues))
            called.append(slot)
            if retry is not None:
                result = retry
        slots[slot] = result
    LAST_SLOT_CALLS = called

    return RankingParams(
        place=_clean_place(slots["place"].place)  # type: ignore[attr-defined]
        if "place" in slots
        else "",
        duration_minutes=slots["duration"].duration_minutes  # type: ignore[attr-defined]
        if "duration" in slots
        else None,
        budget_won=slots["budget"].budget_won  # type: ignore[attr-defined]
        if "budget" in slots
        else None,
        sort_by=slots["sort"].sort_by  # type: ignore[attr-defined]
        if "sort" in slots and slots["sort"].sort_by == "price"  # type: ignore[attr-defined]
        else "distance",
    )


def _extract_by_rule(utterance: str) -> RankingParams:
    """규칙 도구 4종으로 추출합니다. 예외 상황의 폴백 경로입니다."""
    return RankingParams(
        place=extract_place(utterance),
        duration_minutes=extract_duration(utterance),
        budget_won=extract_budget(utterance),
        sort_by=extract_sort(utterance),
    )


def _merge(prev: RankingParams, current: RankingParams) -> RankingParams:
    """직전 조건 위에 이번 발화의 명시 항목만 덮어씁니다.

    sort_by는 RankingParams가 "신호 없음"을 표현할 수 없어
    price(명시 신호)일 때만 덮어쓰고 distance(기본값)면 직전 값을 유지합니다.
    """
    return RankingParams(
        place=current.place if current.place else prev.place,
        duration_minutes=current.duration_minutes
        if current.duration_minutes is not None
        else prev.duration_minutes,
        budget_won=current.budget_won if current.budget_won is not None else prev.budget_won,
        sort_by=current.sort_by if current.sort_by == "price" else prev.sort_by,
        merged_from_previous=True,
    )


# --------------------------------------------------------------------------
# 에이전트 루프용 tool 객체입니다. 파이프라인 내부는 plain 함수를 씁니다.
# --------------------------------------------------------------------------
extract_place_tool = _lc_tool(extract_place)
extract_duration_tool = _lc_tool(extract_duration)
extract_budget_tool = _lc_tool(extract_budget)
extract_sort_tool = _lc_tool(extract_sort)


In [6]:
"""입력 가드레일입니다. [담당: P2]

LLM 호출 전에 규칙으로 차단하여 비용과 지연을 줄입니다.
파라미터 수준의 판정만 수행합니다. 좌표 조회 결과에 따른 분기는
geocode_place 이후에서 처리합니다.
"""

from __future__ import annotations


MAX_DURATION_MINUTES = 24 * 60
MAX_BUDGET_WON = 500_000

INJECTION_PATTERNS = (
    "이전 지시",
    "system prompt",
    "너는 이제",
    "무시하고",
    "개발자 모드",
)


def check_request(params: RankingParams) -> tuple[bool, str | None]:
    """요청을 통과시킬지 판정합니다.

    Returns:
        (통과 여부, 차단 시 안내 문구)
    """
    if not params.place.strip():
        return False, "어느 장소 근처를 찾아 드릴까요? 목적지를 알려주세요."

    lowered = params.place.lower()
    if any(p in lowered for p in INJECTION_PATTERNS):
        return False, "주차장 안내와 관련된 내용만 도와드릴 수 있습니다."

    # 범위 보정은 차단이 아니라 값 조정입니다.
    if params.duration_minutes is not None:
        params.duration_minutes = max(1, min(params.duration_minutes, MAX_DURATION_MINUTES))
    if params.budget_won is not None:
        params.budget_won = max(0, min(params.budget_won, MAX_BUDGET_WON))

    return True, None


## 4. 목적지 좌표와 자치구 (P3)


In [7]:
"""목적지명을 좌표와 자치구로 변환합니다. [담당: P3]

예외를 던지지 않습니다. 실패는 GeocodeResult.message로 표현합니다.
현재 위치는 RequestContext에서 읽습니다. 도구가 GPS를 직접 호출하지 않습니다.
"""

from __future__ import annotations

import math
import os
import requests

KAKAO_API_KEY = os.environ["KAKAO_REST_API_KEY"]
KAKAO_ADDRESS_URL = "https://dapi.kakao.com/v2/local/search/address.json"
KAKAO_KEYWORD_URL = "https://dapi.kakao.com/v2/local/search/keyword.json"


def haversine_m(lat1: float, lng1: float, lat2: float, lng2: float) -> int:
    """두 좌표 사이의 직선 거리를 미터로 반환합니다."""
    r = 6_371_000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lng2 - lng1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return int(2 * r * math.asin(math.sqrt(a)))

def geocode_place(lat: float, lng: float, query: str) -> dict | None:
    """
        장소 이름을 좌표로 변환합니다. 목적지가 언급되면 가장 먼저 호출하세요.
        동명 장소가 여러 곳이면 사용자의 현재 위치에서 가까운 순으로 정렬해 최대 5곳을 반환합니다.
        2곳 이상이면 임의로 고르지 말고 되물으세요.
    """
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {"query": query}
    resp = requests.get(KAKAO_KEYWORD_URL, headers=headers, params=params, timeout=5)
    resp.raise_for_status()
    documents = resp.json().get("documents")
    if not documents:
        return None

    candidates = [
        {
            "lat": float(doc["y"]),
            "lng": float(doc["x"]),
            "address": doc["address_name"],
            "place_name": doc["place_name"],
            "distance_m": haversine_m(lat, lng, float(doc["y"]), float(doc["x"])),
        }
        for doc in documents
    ]
    candidates.sort(key=lambda c: c["distance_m"])
    return candidates[:5]


In [8]:
from __future__ import annotations

SEOUL_DISTRICTS = (
    "강남구",
    "강동구",
    "강북구",
    "강서구",
    "관악구",
    "광진구",
    "구로구",
    "금천구",
    "노원구",
    "도봉구",
    "동대문구",
    "동작구",
    "마포구",
    "서대문구",
    "서초구",
    "성동구",
    "성북구",
    "송파구",
    "양천구",
    "영등포구",
    "용산구",
    "은평구",
    "종로구",
    "중구",
    "중랑구",
)

#: 자기 자신을 첫 원소로 포함
ADJACENT_DISTRICTS: dict[str, list[str]] = {
      "강남구": ["강남구", "서초구", "송파구"],
      "강동구": ["강동구", "송파구"],
      "강북구": ["강북구", "도봉구", "노원구", "성북구", "은평구"],
      "강서구": ["강서구", "양천구", "구로구"],
      "관악구": ["관악구", "동작구", "금천구", "구로구", "서초구"],
      "광진구": ["광진구", "성동구", "동대문구", "중랑구"],
      "구로구": ["구로구", "강서구", "양천구", "영등포구", "금천구"],
      "금천구": ["금천구", "구로구", "관악구", "영등포구"],
      "노원구": ["노원구", "도봉구", "강북구", "중랑구"],
      "도봉구": ["도봉구", "강북구", "노원구"],
      "동대문구": ["동대문구", "성북구", "중랑구", "광진구", "종로구"],
      "동작구": ["동작구", "관악구", "영등포구", "서초구"],
      "마포구": ["마포구", "서대문구", "은평구", "용산구"],
      "서대문구": ["서대문구", "은평구", "종로구", "마포구", "중구"],
      "서초구": ["서초구", "강남구", "동작구", "관악구", "송파구"],
      "성동구": ["성동구", "광진구", "동대문구", "중구", "용산구"],
      "성북구": ["성북구", "강북구", "동대문구", "종로구", "중랑구"],
      "송파구": ["송파구", "강동구", "강남구", "서초구"],
      "양천구": ["양천구", "강서구", "구로구", "영등포구"],
      "영등포구": ["영등포구", "양천구", "강서구", "구로구", "금천구", "동작구"],
      "용산구": ["용산구", "중구", "성동구", "마포구"],
      "은평구": ["은평구", "서대문구", "종로구", "강북구"],
      "종로구": ["종로구", "서대문구", "중구", "성북구", "동대문구"],
      "중구": ["중구", "종로구", "용산구", "성동구", "서대문구"],
      "중랑구": ["중랑구", "노원구", "동대문구", "성북구", "광진구"],
  }


def get_adjacent(district: str) -> list[str]:
    # 자기 자신을 포함해 자치구와 인접 자치구 목록을 반환합니다.
    if district not in SEOUL_DISTRICTS:
        return []
    return ADJACENT_DISTRICTS.get(district, [district])


## 5. 서울시 주차장 조회 (P4)


In [9]:
"""서울시 공영주차장 API 어댑터입니다. [담당: P4]

원본 응답을 ParkingLot으로 변환하는 책임만 집니다.
확인할 수 없는 값은 추정하지 않고 None으로 둡니다.
"""

from __future__ import annotations

import os
import pathlib
import requests

from collections.abc import Iterable



def load_coordinates(code: str) -> tuple[float | None, float | None]:
    """주차장 코드로 좌표를 조회하며, 확인할 수 없으면 None을 반환합니다."""
    key = os.getenv("SEOUL_OPENAPI_KEY")
    if not key:
        return None, None

    timeout_seconds = 10

    try:
        url = (
            f"http://openapi.seoul.go.kr:8088/{key}"
            f"/json/GetParkInfo/1/5/%20/{code}"
        )
        response = requests.get(url, timeout=timeout_seconds)
        response.raise_for_status()
        data = response.json()["GetParkInfo"]

        if data["RESULT"]["CODE"] != "INFO-000":
            return None, None

        for row in data.get("row", []):
            if str(row["PKLT_CD"]) != code:
                continue

            lat = float(row["LAT"])
            lng = float(row["LOT"])

            # 서울 주차장에서 사용할 수 없는 좌표를 제외합니다.
            if not (0 < lat <= 90 and 0 < lng <= 180):
                continue

            return lat, lng

    except Exception:
        return None, None

    return None, None


def load_from_api(districts: Iterable[str]) -> list[ParkingLot]:
    """서울시 실시간 API에서 후보를 읽습니다.

    TODO(P4): GetParkingInfo 연동, 캐시, 좌표 백필을 구현하십시오.
    실패 시 예외를 던지지 말고 빈 리스트를 반환하십시오.
    """
    # 하나의 자치구
    # 키 읽기 -> url 구성 -> 요청 -> 오류 확인 -> JSON 해석 -> GetParkingInfo.RESULT.CODE 확인 → row 목록 추출
    
    # 유효한, 음이 아니 정수만 반환
    def to_int(value: object) -> int | None:
        """유효한 음이 아닌 정수만 반환합니다."""
        try:
            number = float(str(value))
            return int(number) if number >= 0 and number.is_integer() else None
        except (ValueError, OverflowError):
            return None

    # api key
    key = os.getenv("SEOUL_OPENAPI_KEY")

    # 없으면 빈 리스트 반환
    if not key:
        return []

    lots = []
    timeout_seconds = 10
    
    try:
        for district in districts:
            url = (
                f"http://openapi.seoul.go.kr:8088/{key}"
                f"/json/GetParkingInfo/1/100/{district}"
            )
            response = requests.get(url, timeout=timeout_seconds)
            response.raise_for_status()
            data = response.json()["GetParkingInfo"]

            # 예외 처리 (정상 응답이 아니라면 리턴)
            if data["RESULT"]["CODE"] != "INFO-000":
                return []

            # 불러온 주차장 데이터 -> 구조화
            for row in data.get("row", []):
                # 주차장 위도 경도 계산 (API)
                lat, lng = load_coordinates(str(row["PKLT_CD"]))
                lots.append(
                    ParkingLot(
                        code=str(row["PKLT_CD"]),
                        name=row["PKLT_NM"],
                        address=row["ADDR"],
                        district=district,
                        total_slots=to_int(row.get("TPKCT")),
                        current_cars=(
                            to_int(row.get("NOW_PRK_VHCL_CNT"))
                            if str(row.get("PRK_STTS_YN")) == "1"
                            else None
                        ),
                        open_time=row.get("WD_OPER_BGNG_TM"), # 평일기준
                        close_time=row.get("WD_OPER_END_TM"),
                        base_fee=to_int(row.get("BSC_PRK_CRG")),
                        base_minutes=to_int(row.get("BSC_PRK_HR")),
                        extra_fee=to_int(row.get("ADD_PRK_CRG")),
                        extra_minutes=to_int(row.get("ADD_PRK_HR")),
                        lat=lat,
                        lng=lng
                    )
                )
    except Exception:
        return []
    
    return lots


In [10]:
"""목적지 자치구와 인접 자치구의 주차장을 조회합니다. [담당: P4]

예외를 던지지 않습니다. 조회 실패는 빈 목록과 message로 표현합니다.
"""

from __future__ import annotations



def search_parking(destination: Place, ctx: RequestContext) -> SearchResult:
    """목적지가 속한 자치구와 인접 자치구의 공영주차장을 조회합니다.

    geocode_place로 목적지가 확정된 뒤에 호출하세요.
    주차장명, 주소, 시간당 요금, 총 주차면수, 실시간 잔여면을 반환합니다.
    자치구를 직접 지정해 반복 호출하지 마세요.
    """
    districts = get_adjacent(destination.district, ctx.policy.adjacent_district_count)
    if not districts:
        return SearchResult(
            lots=[],
            searched_districts=[],
            message="서울시 자치구가 아니어서 조회할 수 없습니다.",
        )

    lots = load_from_api(districts)

    if not lots:
        return SearchResult(
            lots=[],
            searched_districts=districts,
            message=f"{', '.join(districts)}에서 조회된 주차장이 없습니다.",
        )
    return SearchResult(lots=lots, searched_districts=districts)


## 6. 거리·운영시간·요금 계산 (P5)


In [11]:
"""거리·운영시간·요금을 계산하고 필수조건을 판정합니다. [담당: P5]

이 모듈의 모든 수치는 결정론적으로 계산합니다. LLM을 호출하지 않습니다.
계산할 수 없는 항목은 None으로 두고 사유를 기록합니다.
"""

from __future__ import annotations


MINUTES_PER_DAY = 24 * 60


def evaluate_candidates(
    search: SearchResult,
    destination: Place,
    params: RankingParams,
    ctx: RequestContext,
) -> EvaluationResult:
    """후보별 거리·운영 여부·예상 요금을 계산하고 필수조건으로 거릅니다.

    필수조건을 만족하지 못한 후보는 제외하되 사유를 함께 반환합니다.
    확인할 수 없는 값은 추정하지 않습니다.
    """
    duration = params.duration_minutes or ctx.policy.default_duration_minutes
    assumed = [] if params.duration_minutes else ["duration_minutes"]

    passed: list[Evaluation] = []
    rejected: list[Rejection] = []

    for lot in search.lots:
        reasons: list[RejectReason] = []

        if lot.lat is None or lot.lng is None:
            continue  # 좌표 없는 후보는 거리 계산 불가이므로 조용히 제외합니다.
        distance = haversine_m(destination.lat, destination.lng, lot.lat, lot.lng)
        if distance > ctx.policy.max_distance_m:
            reasons.append("거리 초과")

        is_open, remaining = _check_hours(lot, ctx)
        if not is_open:
            reasons.append("영업 종료")
        elif remaining is not None and remaining < duration:
            reasons.append("영업시간 부족")

        if lot.available_slots is not None and lot.available_slots <= 0:
            reasons.append("만차")

        fee, basis, note = _calculate_fee(lot, duration)
        if params.budget_won is not None and fee is not None and fee > params.budget_won:
            reasons.append("예산 초과")

        if reasons:
            rejected.append(Rejection(lot_name=lot.name, reasons=reasons))
            continue

        passed.append(
            Evaluation(
                lot=lot,
                distance_m=distance,
                is_open=is_open,
                minutes_until_close=remaining,
                estimated_fee=fee,
                fee_basis=basis,
                fee_note=note,
                assumed_fields=list(assumed),
            )
        )

    return EvaluationResult(passed=passed, rejected=rejected)


def _check_hours(lot: ParkingLot, ctx: RequestContext) -> tuple[bool, int | None]:
    """운영 여부와 마감까지 남은 분을 반환합니다.
    """
    if not lot.open_time or not lot.close_time:
        return True, None
    if lot.open_time == "0000" and lot.close_time in ("2400", "0000"):
        return True, None

    now = ctx.request_time.hour * 60 + ctx.request_time.minute
    open_m = int(lot.open_time[:2]) * 60 + int(lot.open_time[2:])
    close_m = int(lot.close_time[:2]) * 60 + int(lot.close_time[2:])

    if open_m < close_m:
        if not (open_m <= now < close_m):
            return False, 0
        return True, close_m - now

    # 자정을 넘기는 운영시간입니다. 예: 22:00~02:00
    if now >= open_m:
        return True, MINUTES_PER_DAY - now + close_m
    if now < close_m:
        return True, close_m - now
    return False, 0


def _calculate_fee(
    lot: ParkingLot, duration_minutes: int
) -> tuple[int | None, str | None, str | None]:
    """예상 요금과 계산 근거를 반환합니다.

    요금 규칙이 없으면 (None, None, 사유)를 반환합니다. 추정하지 않습니다.
    """
    if lot.base_fee is None or lot.base_minutes is None:
        return None, None, "요금 정보 미제공"

    fee = lot.base_fee
    basis = f"기본 {lot.base_minutes}분 {lot.base_fee:,}원"
    extra_minutes = max(0, duration_minutes - lot.base_minutes)

    if extra_minutes and lot.extra_fee and lot.extra_minutes:
        units = -(-extra_minutes // lot.extra_minutes)  # 올림
        fee += units * lot.extra_fee
        basis += f" + 추가 {extra_minutes}분 {units * lot.extra_fee:,}원"
    elif extra_minutes:
        return None, None, "추가 요금 규칙 미제공"

    return fee, basis, "할인·무료시간은 반영되지 않았습니다"


## 7. 정렬·응답·출력 가드레일 (P6)


In [12]:
"""후보를 정렬하고 상위 3곳을 선정합니다. [담당: P6]

정렬은 결정론적으로 수행합니다. LLM에 맡기지 않습니다.
"""

from __future__ import annotations



def rank_candidates(
    evaluation: EvaluationResult,
    params: RankingParams,
    ctx: RequestContext,
) -> RankResult:
    """필수조건을 통과한 후보를 정렬하고 Top 3을 선정합니다.

    추천 목록을 확정하기 직전에 호출하며, 정렬과 선정을 직접 수행하지 마세요.
    조건을 만족하는 후보가 없으면 빈 목록을 반환합니다. 임의로 추천하지 않습니다.
    """
    ordered = sorted(evaluation.passed, key=_sort_key(params.sort_by))
    top = ordered[: ctx.policy.top_k]

    assumed = top[0].assumed_fields if top else []
    return RankResult(
        recommendations=[_to_recommendation(i + 1, e) for i, e in enumerate(top)],
        rejected=evaluation.rejected,
        sort_by=params.sort_by,
        assumed_fields=list(assumed),
    )


def _sort_key(sort_by: str):
    """정렬 기준을 반환합니다.

    잔여 정보가 확인되는 후보를 먼저 배치한 뒤, 선택한 기본 기준의
    확인 불가 값(``None``)을 후순위로 보냅니다. 모든 키가 같은 후보는
    ``sorted``의 안정 정렬에 따라 입력 순서를 유지합니다.
    """

    def key(e: Evaluation):
        availability_unknown = e.lot.available_slots is None
        if sort_by == "price":
            primary_missing = e.estimated_fee is None
            primary = e.estimated_fee if e.estimated_fee is not None else 0
        else:
            primary_missing = False
            primary = e.distance_m
        return (availability_unknown, primary_missing, primary, e.distance_m)

    return key


def _to_recommendation(rank: int, e: Evaluation) -> Recommendation:
    """응답에 노출될 문자열을 완성합니다.

    수치를 문자열로 만드는 책임은 여기까지입니다.
    format_answer는 이 값을 그대로 인용하며 계산하지 않습니다.
    """
    fee_text = f"{e.estimated_fee:,}원" if e.estimated_fee is not None else "계산 불가"

    slots = e.lot.available_slots
    availability_text = f"{slots}면" if slots is not None else "확인 불가"

    if e.minutes_until_close is None:
        hours_text = "24시간"
    else:
        h, m = divmod(e.minutes_until_close, 60)
        close = f"{e.lot.close_time[:2]}:{e.lot.close_time[2:]}" if e.lot.close_time else ""
        hours_text = f"{close} 마감 ({h}시간 {m}분 남음)".strip()

    return Recommendation(
        rank=rank,
        name=e.lot.name,
        distance_m=e.distance_m,
        fee_text=fee_text,
        availability_text=availability_text,
        hours_text=hours_text,
    )


In [13]:
"""최종 응답 문장을 생성합니다. [담당: P6]

이 모듈은 수치를 계산하지 않습니다. Recommendation의 완성된 문자열만 인용합니다.
"""

from __future__ import annotations

import os


FIELD_LABELS = {"duration_minutes": "주차 시간은 1시간 기준"}
REQUIRED_NOTICE = "현재 조회 데이터 기준"


def format_answer(result: RankResult, params: RankingParams, ctx: RequestContext) -> str:
    """추천 결과를 사용자 문장으로 만듭니다."""
    if result.is_empty:
        return _format_empty(result, params)

    if is_llm_disabled():
        return _format_by_template(result, params)
    return _format_by_llm(result, params, ctx) or _format_by_template(result, params)


def _format_empty(result: RankResult, params: RankingParams) -> str:
    """후보 0건 안내입니다. 문구를 고정해 환각 가능성을 차단합니다."""
    if not result.rejected:
        return f"{params.place} 근처에서 조회된 주차장이 없습니다. 다른 목적지로 찾아 드릴까요?"

    counts: dict[str, int] = {}
    for r in result.rejected:
        for reason in r.reasons:
            counts[reason] = counts.get(reason, 0) + 1
    detail = ", ".join(f"{k} {v}곳" for k, v in counts.items())
    return (
        f"{params.place} 근처에서 조건을 만족하는 주차장을 찾지 못했습니다. "
        f"({detail}) 예산을 올리거나 다른 목적지로 다시 찾아 드릴까요?"
    )


def _format_by_template(result: RankResult, params: RankingParams) -> str:
    """템플릿 응답입니다. 키가 없어도 전체 흐름이 검증됩니다."""
    lines = [f"{params.place} 근처 주차장 {len(result.recommendations)}곳입니다."]
    for r in result.recommendations:
        lines.append(
            f"{r.rank}. {r.name} · {r.distance_m}m · {r.fee_text} · "
            f"잔여 {r.availability_text} · {r.hours_text}"
        )
    if result.assumed_fields:
        notes = [FIELD_LABELS.get(f, f) for f in result.assumed_fields]
        lines.append(f"({', '.join(notes)}으로 계산했습니다.)")
    lines.append("현재 조회 데이터 기준이며 실제 현장 상황과 다를 수 있습니다.")
    return "\n".join(lines)


def _format_by_llm(result, params, ctx) -> str | None:
    """LLM으로 설명 문장을 생성합니다.

    LLM은 설명의 자연스러움만 담당하고, 추천 수치의 원천은 ``result``로
    고정합니다. LangChain을 지연 import하므로 규칙 기반 모드와 API 키가
    없는 환경에서도 모듈 import가 실패하지 않습니다.
    """
    try:
        from langchain_core.output_parsers import StrOutputParser
        from langchain_core.prompts import ChatPromptTemplate
        from langchain_openai import ChatOpenAI

        recommendations = "\n".join(
            f"{r.rank}. {r.name} | 거리={r.distance_m}m | 요금={r.fee_text} | "
            f"잔여={r.availability_text} | 운영={r.hours_text}"
            for r in result.recommendations
        )
        prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "당신은 주차장 추천 안내문 작성자입니다.\n"
                    "아래 Recommendation의 값을 그대로 인용하고 숫자를 새로 만들거나 "
                    "반올림하지 마시오. Recommendation에 없는 주차장명, 금액, 거리, "
                    "잔여면, 시각을 추가하지 마시오.\n"
                    f"반드시 '{REQUIRED_NOTICE}' 문구를 포함하시오.\n"
                    "간결한 한국어로 답하고, 추천 후보가 제공한 순서를 유지하시오.\n\n"
                    "Recommendation:\n{recommendations}",
                ),
                ("human", "{place} 근처 주차장 추천을 안내해 주세요."),
            ]
        )
        model = ChatOpenAI(
            model=os.getenv("OPENAI_MODEL", "gpt-5.6-luna"),
            temperature=0,
        )
        answer_chain = prompt | model | StrOutputParser()
        response = answer_chain.invoke(
            {"place": params.place, "recommendations": recommendations}
        )
        return response.strip() if isinstance(response, str) and response.strip() else None
    except Exception:
        return None


In [14]:
"""출력 가드레일입니다. [담당: P6]

답변에 등장한 주차장명·금액·거리·잔여면·마감 시각이 RankResult 안에 있는지 대조합니다.
외부 링크·연락처, 실시간 정보를 확정적으로 표현한 문장도 차단합니다.
LLM 판정 대신 규칙 기반으로 수행해 비용과 지연을 줄입니다.

대조 원칙은 세 가지입니다.
- 표기가 달라도 값이 같으면 통과합니다. ("7800원" = "7,800원" = "7천8백원")
- 값이 조금이라도 다르면 차단합니다. 반올림과 단위 환산 오차도 허용하지 않습니다.
- 추천이 0건이면 답변에 금액·거리·잔여면·시각이 하나도 없어야 합니다.
"""

from __future__ import annotations

import re
from decimal import Decimal, InvalidOperation


SAFE = "SAFE"
UNSAFE = "UNSAFE"

#: 추천이 있는 답변에 반드시 들어가야 하는 안내 문구입니다. (설계서 3.3 실시간 정보 과신 방지)
REQUIRED_NOTICE = "현재 조회 데이터 기준"

#: 실시간 조회 결과를 확정적으로 표현하는 금지 표현입니다.
OVERCONFIDENT_PHRASES = ("보장합니다", "보장됩니다", "확실히", "무조건", "틀림없이", "100%")

#: 이름이 아니라 일반 명사로 쓰이는 '~주차장' 표현입니다. 이름 대조에서 제외합니다.
GENERIC_LOT_WORDS = frozenset(
    {
        "주차장",
        "공영주차장",
        "노상주차장",
        "노외주차장",
        "민영주차장",
        "부설주차장",
        "지하주차장",
        "기계식주차장",
        "유료주차장",
        "무료주차장",
    }
)

#: 판정 사유의 최대 길이입니다. (설계서 2.4 GuardrailVerdict.reason 50자 이내)
MAX_REASON_CHARS = 50
#: 사유에 인용하는 항목의 최대 길이입니다. (설계서 2.4 unsupported_items 20자 이내)
MAX_ITEM_CHARS = 20

_MASK = "§"
_NUM = r"\d+(?:,\d{3})*(?:\.\d+)?"
_KM_UNITS = ("km", "㎞", "킬로미터", "킬로")
_KOREAN_UNITS = (("만", 10_000), ("천", 1_000), ("백", 100))

_LINK = re.compile(r"https?://|www\.|[\w-]+\.(?:com|net|org|kr|io|me|ly)\b", re.IGNORECASE)
_PHONE = re.compile(r"(?<!\d)(?:0\d{1,2}|1\d{3})-\d{3,4}(?:-\d{4})?(?!\d)")
#: '~주차장 근처'처럼 목적지를 가리키는 표현은 주차장명으로 보지 않습니다.
_LOT_NAME = re.compile(r"[가-힣A-Za-z0-9]*주차장(?!\s*(?:근처|주변|인근|앞))")
_WON = re.compile(
    rf"(?:(?:{_NUM})?\s*만\s*)?(?:(?:{_NUM})?\s*천\s*)?(?:(?:{_NUM})?\s*백\s*)?(?:{_NUM})?\s*원"
)
_WON_SYMBOL = re.compile(rf"₩\s*{_NUM}|{_NUM}\s*KRW\b", re.IGNORECASE)
_DISTANCE = re.compile(rf"({_NUM})\s*(km|㎞|킬로미터|킬로|m|미터)(?![A-Za-z])")
_SLOTS = re.compile(rf"({_NUM})\s*(?:면|자리|대)(?!로|학|적|비|당)")
_SLOTS_INVERTED = re.compile(rf"잔여\s*(?:주차\s*)?면수?\s*(?:은|는|이|:)?\s*({_NUM})")
_CLOCK = re.compile(r"(?<!\d)([01]?\d|2[0-4]):([0-5]\d)(?!\d)")


def check_response(answer: str, result: RankResult | None) -> tuple[str, str | None]:
    """답변이 도구 결과에만 근거하는지 판정합니다.

    result가 None이면 랭킹 이전 단계의 고정 안내 문구이므로 대조하지 않습니다.

    Returns:
        ("SAFE" 또는 "UNSAFE", UNSAFE인 경우 50자 이내 사유)
    """
    if result is None:
        return SAFE, None

    violations = collect_violations(answer, result)
    if violations:
        return UNSAFE, violations[0][:MAX_REASON_CHARS]
    return SAFE, None


def collect_violations(answer: str, result: RankResult) -> list[str]:
    """답변에서 도구 결과와 맞지 않는 항목을 모두 찾아 사유 목록으로 반환합니다.

    format_answer가 LLM 답변을 재생성할 때 이 목록을 피드백으로 사용합니다.
    """
    violations: list[str] = []
    allowed_names = [r.name for r in result.recommendations]
    rejected_names = [r.lot_name for r in result.rejected]

    text, mentioned_rejected = _mask_names(answer, allowed_names, rejected_names)
    violations += [_describe("제외된 후보가 언급됨", name) for name in mentioned_rejected]

    violations += [_describe("근거 없는 외부 링크 유도", m.group(0)) for m in _LINK.finditer(text)]
    violations += [_describe("근거 없는 연락처", m.group(0)) for m in _PHONE.finditer(text)]
    violations += [
        _describe("실시간 정보 과신 표현", phrase)
        for phrase in OVERCONFIDENT_PHRASES
        if phrase in text
    ]
    violations += [
        _describe("근거 없는 주차장명", token)
        for token in _LOT_NAME.findall(text)
        if token not in GENERIC_LOT_WORDS
    ]

    fees = {v for r in result.recommendations for _, v in _extract_amounts(r.fee_text)}
    distances = {Decimal(r.distance_m) for r in result.recommendations}
    slots = {v for r in result.recommendations for _, v in _extract_slots(r.availability_text)}
    clocks = {c for r in result.recommendations for c in _extract_clocks(r.hours_text)}

    violations += [
        _describe("근거 없는 금액", raw)
        for raw, value in _extract_amounts(text)
        if value is None or value not in fees
    ]
    violations += [
        _describe("근거 없는 거리", raw)
        for raw, value in _extract_distances(text)
        if value not in distances
    ]
    violations += [
        _describe("근거 없는 잔여면", raw)
        for raw, value in _extract_slots(text)
        if value not in slots
    ]
    violations += [
        _describe("근거 없는 운영시간", clock)
        for clock in _extract_clocks(text)
        if clock not in clocks
    ]

    if not result.is_empty and REQUIRED_NOTICE not in answer:
        violations.append(_describe("안내 문구 누락", REQUIRED_NOTICE))
    return violations


# --------------------------------------------------------------------------
# 주차장명
# --------------------------------------------------------------------------


def _mask_names(
    answer: str, allowed: list[str], rejected: list[str]
) -> tuple[str, list[str]]:
    """알려진 주차장명을 가리고, 답변에 언급된 제외 후보명을 함께 반환합니다.

    이름 속 숫자가 수치 대조에 섞이지 않도록 가립니다. 긴 이름부터 대조하므로
    추천 후보명이 제외 후보명을 포함하는 경우에도 오판하지 않습니다.
    """
    is_allowed: dict[str, bool] = {}
    for name in rejected:
        for variant in _name_variants(name):
            is_allowed.setdefault(variant, False)
    for name in allowed:
        for variant in _name_variants(name):
            is_allowed[variant] = True  # 동명 후보가 양쪽에 있으면 추천 후보로 봅니다.
    if not is_allowed:
        return answer, []

    ordered = sorted(is_allowed, key=len, reverse=True)
    pattern = re.compile("|".join(re.escape(v) for v in ordered))
    mentioned: list[str] = []

    def replace(m: re.Match[str]) -> str:
        if not is_allowed[m.group(0)] and m.group(0) not in mentioned:
            mentioned.append(m.group(0))
        return _MASK

    return pattern.sub(replace, answer), mentioned


def _name_variants(name: str) -> set[str]:
    """이름 표기 변형입니다. 괄호 접미사("(시)")와 공백 생략을 허용합니다."""
    base = re.sub(r"\s*\([^)]*\)\s*$", "", name).strip()
    variants = {name.strip(), base, name.replace(" ", ""), base.replace(" ", "")}
    return {v for v in variants if len(v) >= 2 and v not in GENERIC_LOT_WORDS}


# --------------------------------------------------------------------------
# 수치 추출
# --------------------------------------------------------------------------


def _extract_amounts(text: str) -> list[tuple[str, Decimal | None]]:
    """금액 표현과 원 단위 값을 추출합니다. 해석할 수 없는 금액은 None입니다."""
    found: list[tuple[str, Decimal | None]] = []
    for m in _WON.finditer(text):
        raw = m.group(0).strip()
        if re.search(r"[\d만천백]", raw):  # '공원'처럼 금액이 아닌 '원'은 건너뜁니다.
            found.append((raw, _parse_amount(raw)))
    for m in _WON_SYMBOL.finditer(text):
        raw = m.group(0).strip()
        found.append((raw, _parse_amount(raw)))
    return found


def _parse_amount(raw: str) -> Decimal | None:
    """"1만 5천원" · "7,800원" · "₩7,800" 을 원 단위 값으로 바꿉니다."""
    body = re.sub(r"[\s,원₩]|krw", "", raw, flags=re.IGNORECASE)
    total = Decimal(0)
    try:
        for unit, scale in _KOREAN_UNITS:
            if unit in body:
                head, body = body.split(unit, 1)
                total += (Decimal(head) if head else Decimal(1)) * scale
        if body:
            total += Decimal(body)
    except InvalidOperation:
        return None
    return total


def _extract_distances(text: str) -> list[tuple[str, Decimal]]:
    """거리 표현과 미터 단위 값을 추출합니다."""
    found: list[tuple[str, Decimal]] = []
    for m in _DISTANCE.finditer(text):
        value = _to_decimal(m.group(1))
        if m.group(2) in _KM_UNITS:
            value *= 1000
        found.append((m.group(0), value))
    return found


def _extract_slots(text: str) -> list[tuple[str, Decimal]]:
    """잔여면 표현과 값을 추출합니다. "2면"과 "잔여면 2" 어순을 모두 봅니다."""
    found = [(m.group(0), _to_decimal(m.group(1))) for m in _SLOTS.finditer(text)]
    found += [(m.group(0), _to_decimal(m.group(1))) for m in _SLOTS_INVERTED.finditer(text)]
    return found


def _extract_clocks(text: str) -> list[str]:
    """"HH:MM" 시각을 두 자리 시로 정규화해 추출합니다."""
    return [f"{int(h):02d}:{m}" for h, m in _CLOCK.findall(text)]


def _to_decimal(number: str) -> Decimal:
    return Decimal(number.replace(",", ""))


def _describe(label: str, item: str) -> str:
    return f"{label}: {item.strip()[:MAX_ITEM_CHARS]}"


## 8. LangChain으로 연결하기 (P1)
P2의 슬롯별 구조화 출력으로 조건을 추출하고, P3~P6의 함수를 순서대로 호출합니다.
목적지 후보가 여러 개면 번호를 선택합니다. 후속 요청에는 직전 조건을 전달합니다.

P3의 목록을 공통 자료형으로 변환하고, P4가 사용하는 인접구 개수 인자를 연결합니다.
출력 검증은 `src/pipeline.py`처럼 답변을 수정하지 않고 판정과 사유를 반환합니다.
[RunnableLambda 공식 문서](https://reference.langchain.com/python/langchain-core/runnables/base)


In [15]:
from langchain_core.runnables import RunnableLambda

# P3 원본은 인자 1개, P4 호출부는 인자 2개이므로 연결합니다.
if "p3_get_adjacent" not in globals():
    p3_get_adjacent = get_adjacent

def get_adjacent(district, count=None):
    return p3_get_adjacent(district)


def prepare_request(request):
    ctx = build_context(user_lat=USER_LAT, user_lng=USER_LNG)
    params = extract_params(request["question"], request.get("previous"))
    return {"params": params, "ctx": ctx}


def recommend_parking(state):
    global LAST_DESTINATION, LAST_SEARCH, LAST_EVALUATION, LAST_CONTEXT
    params, ctx = state["params"], state["ctx"]
    ok, message = check_request(params)
    if not ok:
        return AgentResponse(answer=message, params=params)

    try:
        candidates = geocode_place(ctx.user_lat, ctx.user_lng, params.place)
    except requests.RequestException:
        return AgentResponse(answer="목적지 조회에 실패했습니다. 카카오 API 키와 연결을 확인해 주세요.", params=params)
    if not candidates:
        return AgentResponse(answer="목적지를 찾지 못했습니다. 더 구체적인 장소를 알려주세요.", params=params)

    index = 0
    if len(candidates) > 1:
        for i, place in enumerate(candidates, 1):
            print(f"{i}. {place['place_name']} ({place['address']})")
        selection = input("목적지 번호를 선택하세요: ").strip()
        if not selection.isdigit() or not 1 <= int(selection) <= len(candidates):
            return AgentResponse(answer="목적지 번호를 다시 선택해 주세요.", params=params)
        index = int(selection) - 1
    selected = candidates[index]
    address_parts = selected["address"].split()
    district = next((part for part in address_parts if part in SEOUL_DISTRICTS), "")
    if not address_parts or address_parts[0] not in ("서울", "서울시", "서울특별시") or not district:
        return AgentResponse(answer="서울시 목적지만 지원합니다.", params=params)
    destination = Place(
        name=selected["place_name"], address=selected["address"],
        lat=selected["lat"], lng=selected["lng"], district=district,
        distance_from_user_m=selected["distance_m"],
    )
    search = search_parking(destination, ctx)
    evaluation = evaluate_candidates(search, destination, params, ctx)
    # 멀티턴 데모가 선택된 목적지와 계산 결과를 재사용할 수 있도록 보관합니다.
    LAST_DESTINATION, LAST_SEARCH = destination, search
    LAST_EVALUATION, LAST_CONTEXT = evaluation, ctx
    result = rank_candidates(evaluation, params, ctx)
    answer = format_answer(result, params, ctx)
    verdict, reason = check_response(answer, result)
    return AgentResponse(answer, params, result, verdict, reason)


parking_chain = RunnableLambda(prepare_request) | RunnableLambda(recommend_parking)


## 9. 실행하기
서울시 API 조회와 주차장별 좌표 조회로 응답에 시간이 걸릴 수 있습니다.


In [16]:
response = parking_chain.invoke({"question": "강남역 근처 2시간 주차, 만원 이하"})
print(response.answer)
if response.verdict != "SAFE":
    print(f"[가드레일] {response.verdict_reason}")


1. 강남역사거리 (서울 강남구 역삼동)
2. 강남역지하상가 (서울 강남구 역삼동 858)
3. 강남역 2호선 (서울 강남구 역삼동 858)
4. 강남역 신분당선 (서울 강남구 역삼동 858)
5. 강남역센트럴푸르지오시티오피스텔 (서울 강남구 역삼동 825-20)
목적지 번호를 다시 선택해 주세요.


### 직전 조건을 유지하며 가격순으로 다시 찾기


In [17]:
response = parking_chain.invoke({
    "question": "너무 비싸. 저렴한 순으로 찾아줘",
    "previous": response.params,
})
print(response.answer)
if response.verdict != "SAFE":
    print(f"[가드레일] {response.verdict_reason}")


1. 강남역사거리 (서울 강남구 역삼동)
2. 강남역지하상가 (서울 강남구 역삼동 858)
3. 강남역 2호선 (서울 강남구 역삼동 858)
4. 강남역 신분당선 (서울 강남구 역삼동 858)
5. 강남역센트럴푸르지오시티오피스텔 (서울 강남구 역삼동 825-20)
목적지 번호를 다시 선택해 주세요.


### 자유롭게 대화하기


In [32]:
previous = None
while True:
    question = input("질문 (종료: exit): ").strip()
    if question.lower() in ("exit", "quit", "종료"):
        break
    if not question:
        continue
    response = parking_chain.invoke({"question": question, "previous": previous})
    previous = response.params
    print(response.answer)
    if response.verdict != "SAFE":
        print(f"[가드레일] {response.verdict_reason}")


1. 홍대입구역 공항철도 (서울 마포구 동교동 190-1)
2. 홍대입구역 경의중앙선 (서울 마포구 동교동 190-1)
3. 홀리핏 홍대입구역점 (서울 마포구 동교동 174-11)
4. 투루파킹 홍대입구역점 주차장 (서울 마포구 동교동 172-2)
5. 하이그린파킹 홍대입구역점 (서울 마포구 동교동 172-2)
홍대입구역 근처에서 조회된 주차장이 없습니다. 다른 목적지로 찾아 드릴까요?
1. 홍대입구역 공항철도 (서울 마포구 동교동 190-1)
2. 홍대입구역 경의중앙선 (서울 마포구 동교동 190-1)
3. 홀리핏 홍대입구역점 (서울 마포구 동교동 174-11)
4. 투루파킹 홍대입구역점 주차장 (서울 마포구 동교동 172-2)
5. 하이그린파킹 홍대입구역점 (서울 마포구 동교동 172-2)
홍대입구역 근처에서 조회된 주차장이 없습니다. 다른 목적지로 찾아 드릴까요?
1. 홍대입구역 공항철도 (서울 마포구 동교동 190-1)
2. 홍대입구역 경의중앙선 (서울 마포구 동교동 190-1)
3. 홀리핏 홍대입구역점 (서울 마포구 동교동 174-11)
4. 투루파킹 홍대입구역점 주차장 (서울 마포구 동교동 172-2)
5. 하이그린파킹 홍대입구역점 (서울 마포구 동교동 172-2)
홍대입구역 근처에서 조회된 주차장이 없습니다. 다른 목적지로 찾아 드릴까요?


## 10. 실행하기
아래 대화형 실행기는 선택한 목적지와 직전 검색 조건을 유지합니다. 동명 장소의 번호는 처음에만 확인하고, 이후 `더 싼 순으로`, `3시간으로`, `8천 원 이하로` 같은 후속 요청은 기존 목적지와 주차장 후보를 재사용해 재계산·재정렬합니다. 새 목적지를 말했을 때만 목적지를 다시 조회합니다.


In [33]:
class ParkingChat:
    """직전 조건과 사용자가 확정한 목적지를 유지하는 간단한 대화 세션입니다."""

    DISTANCE_WORDS = ("가까운", "거리순", "거리 순", "거리 우선")

    def __init__(self):
        self.params = None
        self.destination = None
        self.search = None
        self.pending_candidates = None
        self.pending_params = None

    def ask(self, question: str) -> AgentResponse:
        question = question.strip()

        # 동명 장소 확인은 대화의 다음 턴으로 처리합니다.
        if self.pending_candidates is not None:
            if question.isdigit():
                return self._choose_destination(int(question) - 1)
            return AgentResponse(
                answer=f"1~{len(self.pending_candidates)} 중 목적지 번호를 입력해 주세요.",
                params=self.pending_params,
            )

        previous_place = self.params.place if self.params else None
        params = extract_params(question, self.params)
        if any(word in question for word in self.DISTANCE_WORDS):
            params.sort_by = "distance"

        ok, message = check_request(params)
        if not ok:
            return AgentResponse(answer=message, params=params)

        # 목적지가 그대로면 직전 후보를 새 조건으로만 재평가합니다.
        same_destination = self.destination is not None and params.place == previous_place
        self.params = params
        if same_destination:
            return self._recommend(params)

        self.destination = None
        self.search = None
        ctx = build_context(user_lat=USER_LAT, user_lng=USER_LNG)
        try:
            candidates = geocode_place(ctx.user_lat, ctx.user_lng, params.place)
        except requests.RequestException:
            return AgentResponse(
                answer="목적지 조회에 실패했습니다. 카카오 API 키와 연결을 확인해 주세요.",
                params=params,
            )
        if not candidates:
            return AgentResponse(
                answer="목적지를 찾지 못했습니다. 더 구체적인 장소를 알려주세요.",
                params=params,
            )
        if len(candidates) > 1:
            self.pending_candidates = candidates
            self.pending_params = params
            lines = ["동명 장소가 여러 곳입니다. 목적지 번호를 선택해 주세요."]
            lines += [
                f"{i}. {place['place_name']} ({place['address']})"
                for i, place in enumerate(candidates, 1)
            ]
            return AgentResponse(answer="\n".join(lines), params=params)
        return self._set_destination(candidates[0], params, ctx)

    def _choose_destination(self, index: int) -> AgentResponse:
        if not 0 <= index < len(self.pending_candidates):
            return AgentResponse(
                answer=f"1~{len(self.pending_candidates)} 중에서 선택해 주세요.",
                params=self.pending_params,
            )
        selected = self.pending_candidates[index]
        params = self.pending_params
        self.pending_candidates = None
        self.pending_params = None
        ctx = build_context(user_lat=USER_LAT, user_lng=USER_LNG)
        return self._set_destination(selected, params, ctx)

    def _set_destination(self, selected, params, ctx) -> AgentResponse:
        parts = selected["address"].split()
        district = next((part for part in parts if part in SEOUL_DISTRICTS), "")
        if not parts or parts[0] not in ("서울", "서울시", "서울특별시") or not district:
            return AgentResponse(answer="서울시 목적지만 지원합니다.", params=params)

        self.destination = Place(
            name=selected["place_name"],
            address=selected["address"],
            lat=selected["lat"],
            lng=selected["lng"],
            district=district,
            distance_from_user_m=selected["distance_m"],
        )
        self.search = search_parking(self.destination, ctx)
        return self._recommend(params, ctx)

    def _recommend(self, params, ctx=None) -> AgentResponse:
        ctx = ctx or build_context(user_lat=USER_LAT, user_lng=USER_LNG)
        evaluation = evaluate_candidates(self.search, self.destination, params, ctx)
        result = rank_candidates(evaluation, params, ctx)
        answer = format_answer(result, params, ctx)
        verdict, reason = check_response(answer, result)
        return AgentResponse(answer, params, result, verdict, reason)


In [35]:
chat = ParkingChat()
print("예: '강남역 근처 2시간 주차, 만원 이하' → 목적지 번호 → '더 싼 순으로'")

while True:
    question = input("\n나 > ").strip()
    if question.lower() in ("exit", "quit", "종료"):
        print("대화를 종료합니다.")
        break
    if not question:
        continue

    response = chat.ask(question)
    print(f"\n여긴어때 > {response.answer}")
    if response.verdict != "SAFE":
        print(f"[가드레일] {response.verdict_reason}")


예: '강남역 근처 2시간 주차, 만원 이하' → 목적지 번호 → '더 싼 순으로'

여긴어때 > 동명 장소가 여러 곳입니다. 목적지 번호를 선택해 주세요.
1. 강남역사거리 (서울 강남구 역삼동)
2. 강남역지하상가 (서울 강남구 역삼동 858)
3. 강남역 2호선 (서울 강남구 역삼동 858)
4. 강남역 신분당선 (서울 강남구 역삼동 858)
5. 강남역센트럴푸르지오시티오피스텔 (서울 강남구 역삼동 825-20)

여긴어때 > 현재 조회 데이터 기준 강남역 근처 주차장 추천입니다.

1. 양재역 공영주차장(시): 거리 1710m, 요금 19,200원, 잔여 648면, 24시간 운영
2. 반포천 공영주차장(파미에)(시): 거리 1885m, 요금 26,400원, 잔여 160면, 24시간 운영

여긴어때 > 현재 조회 데이터 기준, 강남역 근처 추천 주차장은 다음과 같습니다.

1. 양재역 공영주차장(시): 거리 1710m, 요금 19,200원, 잔여 648면, 24시간 운영
2. 반포천 공영주차장(파미에)(시): 거리 1885m, 요금 26,400원, 잔여 160면, 24시간 운영

여긴어때 > 동명 장소가 여러 곳입니다. 목적지 번호를 선택해 주세요.
1. 청와대소금구이 (서울 강남구 삼성동 140-32)
2. 청와대손만두 (서울 동대문구 장안동 105-15)
3. 인생네컷 청와대직영점 (서울 종로구 창성동 35)
4. 포비 청와대점 (서울 종로구 팔판동 115-52)
5. 공근혜갤러리 (서울 종로구 삼청동 157-78)

여긴어때 > 청와대 근처에서 조회된 주차장이 없습니다. 다른 목적지로 찾아 드릴까요?
대화를 종료합니다.
